# WELL LOG ANALYZER


## INDEX

1. [Importing](#1)
2. [--](#2)

<a name="1"></a>
### Importing

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import scipy
import os
from docx import Document
from docx.shared import Inches
from scipy.optimize import minimize
import tkinter as tk
from tkinter import filedialog
from IPython.display import display
import os
from matplotlib.backends.backend_pdf import PdfPages
import lasio

In [10]:
def read_log(file_path, file_name)->pd.DataFrame:
    full_path = os.path.join(file_path, file_name)
    _, ext = os.path.splitext(file_name.lower())
    try:
        if ext == ".csv":
            df = pd.read_csv(full_path)
        elif ext == ".las":
            lf = lasio.read(full_path)
            df = lf.df().reset_index()
        elif ext in ['.xls','.xlsx']:
            df = pd.read_excel(full_path)
        print("Printing description of dataset:\n" + str(df.describe()))
        print("Printing available logs:\n" + str(df.columns))
        df = df.replace([-999, -999.25, -999.5, -999.75], np.nan)
        return df
    except Exception as e:
        print(f"Unsupported File Format!\nError: {e}")
        return None


In [20]:
file_path = r"C:\Users\Soumik Dutta\Documents\Petroleum\13Data Science\Well Log Analyzer\data\raw"
file_name = r"Example.LAS"
df = read_log(file_path, file_name)

Printing description of dataset:
              DEPTH     ABDCQF01     ABDCQF02     ABDCQF03     ABDCQF04  \
count  33191.000000  3226.000000  3226.000000  3226.000000  3226.000000   
mean    1805.400000     2.449846     2.447409     2.447272     2.449575   
std      958.156073     0.131558     0.130183     0.131161     0.132434   
min      145.900000     2.104600     2.118200     2.090200     2.124700   
25%      975.650000     2.366375     2.359250     2.361750     2.360050   
50%     1805.400000     2.479150     2.478850     2.478250     2.479650   
75%     2635.150000     2.527275     2.526050     2.525200     2.526225   
max     3464.900000     3.057700     3.052100     3.050500     3.086600   

                 BS         CALI         DRHO           DT          DTS  ...  \
count  33191.000000  3519.000000  3526.000000  4262.000000  3809.000000  ...   
mean      19.043252     8.677213     0.053590    80.380006   138.251061  ...   
std        6.613720     0.083978     0.023488    14

In [21]:
df.to_csv(r"C:\Users\Soumik Dutta\Documents\Petroleum\13Data Science\Well Log Analyzer\data\processed\Example.csv", index=False)

In [26]:
def cleaning_log(df, valid_logs = None)->pd.DataFrame:
    try:
        try:
            df = df[valid_logs]
        except Exception as e:
            print(f"Error: {e}\nProceeding with all available logs.")
        df = df.dropna(axis = 0)
        print("Printing description of cleaned dataset:\n" + str(df.describe()))
        return df
    except Exception as e:
        print(f"Error: {e}")


In [28]:
df = cleaning_log(df, ['DEPTH','BS', 'CALI', 'GR', 'RM', 'RD', 'RT', 'RHOB', 'NPHI', 'DRHO', 'DT', 'DTS', 'PEF'])

Printing description of cleaned dataset:
             DEPTH      BS         CALI           GR           RM  \
count  2813.000000  2813.0  2813.000000  2813.000000  2813.000000   
mean   3272.211234     8.5     8.667867    51.788297     4.476485   
std      96.909413     0.0     0.038018    27.740720    14.075433   
min    3098.500000     8.5     8.360900     8.001500     0.234900   
25%    3205.200000     8.5     8.663700    36.332600     1.339300   
50%    3284.000000     8.5     8.663700    47.600200     1.670200   
75%    3354.300000     8.5     8.697600    57.254100     2.475700   
max    3424.600000     8.5     8.799100   160.992400   134.704100   

                RD           RT         RHOB         NPHI         DRHO  \
count  2813.000000  2813.000000  2813.000000  2813.000000  2813.000000   
mean      3.527763     4.476485     2.471040     0.174779     0.051649   
std       9.242271    14.075399     0.121311     0.057355     0.012793   
min       0.373400     0.234900     2.150

In [ ]:
def gr_constant_selector(gr_values, depth_value, constant_selector = None):
    if constant_selector is None:
        try:
            gr_shale = np.max(gr_values)
            gr_sand = np.min(gr_values)         
        except Exception as e:
            print(f"Error calculating shale value: {e}")
            gr_shale = np.nan
            gr_sand = np.nan
    elif constant_selector == 'averaged:
        gr_shale = constant_selector

    return gr_shale, gr_sand

def calculate_GR(df, gr_col = 'GR', depth_col = 'DEPTH', constant_selector = None, shale_model = None):
    try:
        if gr_col not in df.columns or depth_col not in df.columns:
            raise ValueError(f"Columns '{gr_col}' and/or '{depth_col}' not found in DataFrame.")
        gr_values = df[gr_col].values
        depth_values = df[depth_col].values
        return gr_values, depth_values
    except Exception as e:
        print(f"Error: {e}")
        return None, None